---
title: "Data Transformation"
format:
  html:
    embed-resources: true
    code-fold: false
    toc: true

---

## Overview

This section covers the methodology of how we performed some simple one-hot encoding and aggregation on each row of our filtered dataset, in order to more smoothly perform the next steps.

## State of the Data

After running our Job to get and clean the data from the Reddit corpus (as described in the "Initial Filtering & EDA" tab), we were still working with a significant amount of data: 

![](images/afterCleanCounts.png)

About five million submissions and just over 74 million comments, each occupying their own row in two separate data sets. We knew that all these posts had been made in our relevant time period, contained text in English, were not from our excluded subreddits, and - most importantly! - contained at least one of our targetted keywords.

#### Our Keywords

A quick note about our keywords! These were created by the team after several discussions on what we did or did not think would be relevant, and what attached sentiment the post surrounding the keyword might indicate. Our keywords were then broken into three categories: 

1) right-leaning, where positive sentiment containing these keywords would indicate preference towards Republicans, and negative sentiment would indicate Democrats

2) left-leaning, the opposite of right-leaning, where positive sentiment would indicate support for the Democrats, and vice versa

3) generic, where presence of the terms themselves indicates that the posts is discussing a politically charged subject, but that may or may not have a strong correlation with a particular party

The inclusion of generic keywords was a hotly debated topic among our group, even as we progressed in our data analysis. You can see at later steps that we revisted the list of keywords in order to assign some lean to several "even more key" keywords. However, for the step below, the summation columns were calculated with the original three lists.

## One-hot encoding (by hand!)

Now that we had our Reddit data with relevant keywords, it was time to transform our text posts into something that could be more easily quantified, in order to allow us to label the still-massive dataset quickly. The script below was submitted as a Job in Azure ML in order to create columns containing boolean (binary) data indicating if that keyword was observed in the data. After the one hot encoding was performed concerning keywords, summation columns were added to indicate the presence of keywords from each list, and how many keywords (if any) were present. This was initially tested on our smaller r/politics dataset, then submitted as a job on the Reddit-wide scraped data.  Here you can see an example of the resulting count after this was run on the (much smaller!) r/politics set: 

![](images/politicscount.png)

Below is the script that performed this transformation - behold!

## The Script

In [ ]:

### Libraries and session start ###

from azureml.core import Workspace, Dataset, Datastore
import sparknlp
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .config("spark.jars.packages", "com.johnsnowlabs.nlp:spark-nlp_2.12-5.0.2") \
    .getOrCreate()

### Initial data read ###

#read data
subscription_id = 'TEMPLATE' #removed for security reasons
resource_group = 'project-group-11'
workspace_name = 'project-group-11'
workspace = Workspace(subscription_id, resource_group, workspace_name)
datastore = Datastore.get(workspace, "workspaceblobstore")

# Read data and repartition immediately if large data volumes are expected
comments_df = Dataset.Tabular.from_parquet_files(path=(datastore, 'sampledata/clean_comments.parquet')).to_spark_dataframe().repartition(200)
submissions_df = Dataset.Tabular.from_parquet_files(path=(datastore, 'sampledata/clean_submissions.parquet')).to_spark_dataframe().repartition(200)

#submissions_df.head()

### Keyword futzing ###

# Function to load the keywords
def load_keywords(file_path):
    with open(file_path, 'r') as f:
        return [line.strip() for line in f.readlines()]

# Load all keywords, skipping lemmatization
keywords_left_lemmatized = load_keywords("Keywords_Left_Lemmatized.txt")
keywords_right_lemmatized = load_keywords("Keywords_Right_Lemmatized.txt")
keywords_non_partisan_lemmatized = load_keywords("Keywords_Non_Partisan_Lemmatized.txt")
keywords_left_unchanged = load_keywords("Keywords_Left_Unchanged.txt")
keywords_right_unchanged = load_keywords("Keywords_Right_Unchanged.txt")
keywords_non_partisan_unchanged = load_keywords("Keywords_Non_Partisan_Unchanged.txt")

#minor tweaks
keywords_left_unchanged = keywords_left_unchanged + ["Democrat", "Democratic"]
keywords_right_unchanged = keywords_right_unchanged + ["Republican", "Republicans", "defund the police", "red flag laws", "critical race theory"]

# Combine all keywords into a single list
all_keywords = set(
    keywords_left_unchanged + keywords_right_unchanged + keywords_non_partisan_unchanged +
    keywords_left_lemmatized + keywords_right_lemmatized + keywords_non_partisan_lemmatized
)

#more minor tweaks
#the below fields will never be found since they're part of labelling
all_keywords.remove("CRT (critical race theory)")
all_keywords.remove("defund the police (criticized)")
all_keywords.remove("defund the police (advocated)")
all_keywords.remove("red flag laws (supported)")
all_keywords.remove("red flag laws (opposed)")

### The Meat - creates yes/no columns for each keyword ###

#change to list so can batch this bad boy UP
all_keywords = list(all_keywords)

#function to match keywords in a batch
#comments version
def process_batch_comments(keywords_batch):
    keyword_columns = [
        F.when(F.col('body').contains(keyword), 1).otherwise(0).alias(keyword)
        for keyword in keywords_batch
    ]
    return comments_df.select('*', *keyword_columns)

#submission version
def process_batch_submissions(keywords_batch):
    keyword_columns = [
        F.when(F.col('selftext').contains(keyword), 1).otherwise(0).alias(keyword)
        for keyword in keywords_batch
    ]
    return submissions_df.select('*', *keyword_columns)

# Split all keywords into smaller batches
# this is why you changed it to a list!
batch_size = 1000
batches = [all_keywords[i:i + batch_size] for i in range(0, len(all_keywords), batch_size)]

# Process each batch and combine results
for batch in batches:
    comments_df = process_batch_comments(batch)
    submissions_df = process_batch_submissions(batch)


#adding summation columns
comments_df_weights = (
    comments_df
    .withColumn("ContainsLeft", F.greatest(*[F.col(col) for col in left_keywords]))
    .withColumn("ContainsRight", F.greatest(*[F.col(col) for col in right_keywords]))
    .withColumn("ContainsGeneric", F.greatest(*[F.col(col) for col in generic_keywords]))
    .withColumn("SumLeft", sum(F.col(col) for col in left_keywords))
    .withColumn("SumRight", sum(F.col(col) for col in right_keywords))
    .withColumn("SumGeneric", sum(F.col(col) for col in generic_keywords))
)

submissions_df = submissions_df.withColumnRenamed("Roe v. Wade", "Roe_v_Wade")
submissions_df_weights = (
    submissions_df
    .withColumn("ContainsLeft", F.greatest(*[F.col(col) for col in left_keywords]))
    .withColumn("ContainsRight", F.greatest(*[F.col(col) for col in right_keywords]))
    .withColumn("ContainsGeneric", F.greatest(*[F.col(col) for col in generic_keywords]))
    .withColumn("SumLeft", sum(F.col(col) for col in left_keywords))
    .withColumn("SumRight", sum(F.col(col) for col in right_keywords))
    .withColumn("SumGeneric", sum(F.col(col) for col in generic_keywords))
)

### Write to file ###

submissions_df_weights.write.mode("overwrite").parquet("azureml://datastores/workspaceblobstore/paths/cleandata/submissions_w_keywords_and_weights.parquet")
comments_df_weights.write.mode("overwrite").parquet("azureml://datastores/workspaceblobstore/paths/cleandata/comments_w_keywords_and_weights.parquet")